## Imports

In [1]:
import os
import re
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

In [3]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [5]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/analytics/customer/Ads-Monetisation/latency-experiment'
print(local_datasource)

/Users/E2074/analytics/customer/Ads-Monetisation/latency-experiment
/Users/E2074/analytics/customer/Ads-Monetisation/latency-experiment


## Datasets & Parameter

In [6]:
## Parameter 
current_date = datetime.now()
start_date = '20240901'
end_date = '20240901'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240901  to  20240901


In [21]:
Ad_Request_query = f"""

    WITH request as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        CASE WHEN appversion <= '8.28.0' THEN '8.28.0 >=' ELSE appversion END appversion,
        isprefetched,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch 
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '{start_date}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end_date}' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        -- and [[ {{city}} --]] city IS NOT NULL
        and regexp_like(lower(slotname), 'ontheway')
        and eventName = 'Ad_Request'
    )

    SELECT * FROM request
"""

# Execute the query and get the result as a DataFrame
df_Ad_Request = pd.read_sql(Ad_Request_query, connection)

In [22]:
df_Ad_Request

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,03,Bangalore,8.30.0,None,638979580feaab25b46e31f4,PostOrderOnTheWay3,gamCombinedBanner,ad_638979580feaab25b46e31f4_1725103310941,1725141055658
1,20240901,01,Bangalore,8.30.0,true,5e48ace9b3394b868d2a0272,PostOrderOnTheWay1,OTWS0,8eda7e97-0a04-4ca9-bc6a-b4d1ecad094f5e48ace9b3...,1725135835998
2,20240901,03,Bangalore,8.30.0,None,6191a665c98cb3208affbc7e,PostOrderOnTheWay3,gamCombinedBanner,ad_6191a665c98cb3208affbc7e_1725142049033,1725142102367
3,20240901,02,Bangalore,8.29.2,None,5fa0e5280c234d9a4fa85213,PostOrderOnTheWay3,gamCombinedBanner,ad_5fa0e5280c234d9a4fa85213_1725137390029,1725137424717
4,20240901,04,Bangalore,8.30.0,None,6253864ffc8d84aec2afda9a,PostOrderOnTheWay3,gamCombinedBanner,ad_6253864ffc8d84aec2afda9a_1725143363215,1725146488771
...,...,...,...,...,...,...,...,...,...,...
1333391,20240901,21,Bangalore,8.30.0,true,655f7e7834e8651dae006c25,PostOrderOnTheWay1,OTWS0,4d610e23-d8cc-47ab-a165-0aa2d70b604d655f7e7834...,1725204770415
1333392,20240901,15,Bangalore,8.30.0,None,659bfbd1b130a9c170c28616,PostOrderOnTheWay1,gamCombinedBanner,ad_659bfbd1b130a9c170c28616_1725164805273,1725185392653
1333393,20240901,15,Bangalore,8.30.0,None,66d31d9909eb4a96b2b7e482,PostOrderOnTheWay1,gamCombinedBanner,ad_66d31d9909eb4a96b2b7e482_1725184305297,1725184385187
1333394,20240901,15,Bangalore,8.30.0,None,62541c3a6bfbf362e78427d8,PostOrderOnTheWay1,gamCombinedBanner,ad_62541c3a6bfbf362e78427d8_1725184805358,1725184869331


In [23]:
df_Ad_Request.slotname.unique()

array(['PostOrderOnTheWay3', 'PostOrderOnTheWay1', 'PostOrderOnTheWay2'],
      dtype=object)

In [30]:
Ad_Response_query= f"""

    WITH response as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        appversion,
        isprefetched,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch 
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '{start_date}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end_date}' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        -- and [[ {{city}} --]] city IS NOT NULL
        and regexp_like(lower(slotname), 'ontheway')
        and eventName = 'Ad_Response'
    )

    SELECT * FROM response    
"""

# Execute the query and get the result as a DataFrame
df_Ad_Response = pd.read_sql(Ad_Response_query, connection)

In [31]:
df_Ad_Response

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,01,Bangalore,8.30.0,None,5c6f95af35478b5602791ea9,PostOrderOnTheWay1,gamCombinedBanner,ad_5c6f95af35478b5602791ea9_1724678325647,1725135016417
1,20240901,07,Bangalore,8.30.0,true,62cc2cf7551cdbfd0dc2b31a,PostOrderOnTheWay1,OTWS0,4a18b54b-99a3-4beb-8c77-4ef1ea51bb6262cc2cf755...,1725156669978
2,20240901,07,Bangalore,8.30.0,true,6141b2c97f5ff74fe40629ea,PostOrderOnTheWay2,OTWS1,460d5914-ceeb-4e87-90aa-2c21583256f36141b2c97f...,1725155516180
3,20240901,00,Bangalore,8.30.0,None,62601bb5e98a646af092477c,PostOrderOnTheWay1,gamCombinedBanner,ad_62601bb5e98a646af092477c_1722935251659,1725129457269
4,20240901,22,Bangalore,8.30.0,true,64ad57939c5afe6e3daf3a6d,PostOrderOnTheWay2,OTWS1,1e0bc859-a19a-4440-a716-378dac01bc7064ad57939c...,1725208843787
...,...,...,...,...,...,...,...,...,...,...
683171,20240901,09,Bangalore,8.30.0,true,617a9f8369ebf84da6e242e1,PostOrderOnTheWay1,OTWS0,7f090941-bfd2-4576-9c3c-c1f852f7d4a7617a9f8369...,1725162915327
683172,20240901,01,Bangalore,8.30.0,true,5c720597343a5357615b1d38,PostOrderOnTheWay2,OTWS1,289d8874-d619-4294-bd4a-cd01a66181155c72059734...,1725134711526
683173,20240901,16,Bangalore,8.30.0,true,639c4b03aa6416eca855c6cd,PostOrderOnTheWay1,OTWS0,b559b08e-5912-49f7-9cf1-1a251f84e079639c4b03aa...,1725188659838
683174,20240901,15,Bangalore,8.30.0,true,6287c24f8f9b4359095f51a1,PostOrderOnTheWay2,OTWS1,2a4e5927-6c00-485a-8815-cfd57fad4dda6287c24f8f...,1725186301700


In [32]:
df_Ad_Response.slotname.unique()

array(['PostOrderOnTheWay1', 'PostOrderOnTheWay2', 'PostOrderOnTheWay3'],
      dtype=object)

In [34]:
Ad_Impression_query= f"""

    WITH impression as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        appversion,
        isprefetched,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch 
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '{start_date}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end_date}' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        -- and [[ {{city}} --]] city IS NOT NULL
        and regexp_like(lower(slotname), 'ontheway')
        and eventName = 'Ad_Impression'
    )

    SELECT * FROM impression    
"""

# Execute the query and get the result as a DataFrame
df_Ad_Impression = pd.read_sql(Ad_Impression_query, connection)

In [35]:
df_Ad_Impression

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,00,Bangalore,8.30.0,None,653e461fe3ed5c41697f1c78,PostOrderOnTheWay1,gamCombinedBanner,ad_653e461fe3ed5c41697f1c78_1725044677158,1725130987570
1,20240901,08,Bangalore,8.30.0,true,60e3195c09aa99e28940772b,PostOrderOnTheWay1,OTWS0,ae8f7109-aa55-4a85-924e-5302e966be5460e3195c09...,1725159746591
2,20240901,10,Bangalore,8.29.2,None,657fd3ca7839e236debee4e7,PostOrderOnTheWay1,gamCombinedBanner,ad_657fd3ca7839e236debee4e7_1725164906946,1725165129191
3,20240901,13,Bangalore,8.29.2,None,63ca56d9fad55c621d33955b,PostOrderOnTheWay2,gamCombinedBanner,ad_63ca56d9fad55c621d33955b_1725178896127,1725179076656
4,20240901,14,Bangalore,8.30.0,true,613c499dddee0ba27d671669,PostOrderOnTheWay1,OTWS0,d794672c-23a0-420b-892d-053c2cdb24ba613c499ddd...,1725180628747
...,...,...,...,...,...,...,...,...,...,...
75394,20240901,19,Bangalore,8.30.0,true,6558c11ebc13b94951c04e8b,PostOrderOnTheWay1,OTWS0,1725fd3c-acd3-4f8c-ae1e-e12baf1d984f6558c11ebc...,1725197823310
75395,20240901,16,Bangalore,8.30.0,true,5d1e2edb93686945e712c2ee,PostOrderOnTheWay2,OTWS1,4ec8e812-1a75-43a9-889a-47d677a58ccc5d1e2edb93...,1725187944363
75396,20240901,18,Bangalore,8.29.2,None,6035ffb0f6ea6bb4d01a8c19,PostOrderOnTheWay1,gamCombinedBanner,ad_6035ffb0f6ea6bb4d01a8c19_1725153098347,1725194588405
75397,20240901,18,Bangalore,8.30.0,true,633ac14de90ec6f88e7d63e5,PostOrderOnTheWay1,OTWS0,239c9994-a4ca-454e-ad5d-3f5a5f51ec82633ac14de9...,1725197018225


In [36]:
df_Ad_Impression.slotname.unique()

array(['PostOrderOnTheWay1', 'PostOrderOnTheWay2', 'PostOrderOnTheWay3'],
      dtype=object)

In [37]:
Ad_viewable_impression_query= f"""

    WITH viewable_impression as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        appversion,
        isprefetched,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch 
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '{start_date}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end_date}' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        -- and [[ {{city}} --]] city IS NOT NULL
        and regexp_like(lower(slotname), 'ontheway')
        and eventName = 'Ad_Viewable_Impression'
    )

    SELECT * FROM viewable_impression    
    
"""

# Execute the query and get the result as a DataFrame
df_Ad_viewable_impression = pd.read_sql(Ad_viewable_impression_query, connection)

In [38]:
df_Ad_viewable_impression

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,06,Bangalore,8.30.0,None,63085e5b5f964737deea5f56,PostOrderOnTheWay1,gamCombinedBanner,ad_63085e5b5f964737deea5f56_1724315142707,1725152381745
1,20240901,05,Bangalore,8.29.2,None,5e32dab14cb0d02b712dbc7c,PostOrderOnTheWay1,gamCombinedBanner,ad_5e32dab14cb0d02b712dbc7c_1724852928577,1725147545082
2,20240901,02,Bangalore,8.26.0,None,5e3e7150aa5b4d79daa41810,PostOrderOnTheWay1,gamCombinedBanner,ad_5e3e7150aa5b4d79daa41810_1716543443971,1725137408748
3,20240901,05,Bangalore,8.30.0,None,62381c49de914834ed229bee,PostOrderOnTheWay1,gamCombinedBanner,ad_62381c49de914834ed229bee_1725149362593,1725149382554
4,20240901,09,Bangalore,8.30.0,None,667a30d8f889a77ed244ad5c,PostOrderOnTheWay2,gamCombinedBanner,ad_667a30d8f889a77ed244ad5c_1724477918209,1725161676311
...,...,...,...,...,...,...,...,...,...,...
107175,20240901,10,Bangalore,8.30.0,None,653539a2911e525036cf6422,PostOrderOnTheWay1,gamCombinedBanner,ad_653539a2911e525036cf6422_1725104118025,1725166456248
107176,20240901,10,Bangalore,8.30.0,None,65ca78d72b682d5b1fbe613e,PostOrderOnTheWay1,gamCombinedBanner,ad_65ca78d72b682d5b1fbe613e_1725165280666,1725165307982
107177,20240901,10,Bangalore,8.30.0,None,651d5080440a42d9fdf6963b,PostOrderOnTheWay1,gamCombinedBanner,ad_651d5080440a42d9fdf6963b_1725167232794,1725167385163
107178,20240901,15,Bangalore,8.29.2,None,66500ca2d9106ab2573104e4,PostOrderOnTheWay1,gamCombinedBanner,ad_66500ca2d9106ab2573104e4_1725001313730,1725185037759


In [39]:
Ad_click_query= f"""

    WITH click as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        appversion,
        isprefetched,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch 
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '{start_date}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end_date}' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        -- and [[ {{city}} --]] city IS NOT NULL
        and regexp_like(lower(slotname), 'ontheway')
        and eventName = 'Ad_Click'
    )

    SELECT * FROM click    
    
"""

# Execute the query and get the result as a DataFrame
df_Ad_click = pd.read_sql(Ad_click_query, connection)

In [40]:
df_Ad_click

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,21,Bangalore,8.30.0,true,65472c506305b2dd42d4cfa9,PostOrderOnTheWay1,OTWS0,cce203b4-35d2-4bd8-a816-8af48c1749e465472c5063...,1725205455685
1,20240901,10,Bangalore,8.30.0,true,64dc4fb06e299fde024afc38,PostOrderOnTheWay1,OTWS0,3c848ead-179a-4c8c-a993-5c93dd714a6864dc4fb06e...,1725167322514
2,20240901,17,Bangalore,8.29.2,None,657b4dd7a55bd42023391852,PostOrderOnTheWay1,gamCombinedBanner,ad_657b4dd7a55bd42023391852_1725191136909,1725191508550
3,20240901,20,Bangalore,8.30.0,true,63293d919a4a212016f69eee,PostOrderOnTheWay1,OTWS0,c09cc989-b265-4c4c-bb48-8dac4b0ac7fa63293d919a...,1725202089019
4,20240901,20,Bangalore,8.30.0,None,61c22acff3de6f18d1531eb6,PostOrderOnTheWay1,gamCombinedBanner,ad_61c22acff3de6f18d1531eb6_1725201346979,1725201615955
...,...,...,...,...,...,...,...,...,...,...
286,20240901,11,Bangalore,8.30.0,true,64d24bbefd756865ba8385ae,PostOrderOnTheWay2,OTWS1,d881d274-5b78-4dd3-81d7-77fb5182661864d24bbefd...,1725170043869
287,20240901,05,Bangalore,8.30.0,true,633b8e8c8592da03fafdad68,PostOrderOnTheWay1,OTWS0,afb943b3-23cc-4a69-8c20-260de63c1c9e633b8e8c85...,1725150048015
288,20240901,08,Bangalore,8.30.0,None,5a4d9de59d91067e52c28bbc,PostOrderOnTheWay1,gamCombinedBanner,ad_5a4d9de59d91067e52c28bbc_1725158034999,1725158167888
289,20240901,10,Bangalore,8.29.2,None,5d8ee7d8941a7d1c51c8bd2f,PostOrderOnTheWay1,gamCombinedBanner,ad_5d8ee7d8941a7d1c51c8bd2f_1724509005222,1725167134669


In [46]:
df_Ad_click[df_Ad_click['aduuid'] == 'cce203b4-35d2-4bd8-a816-8af48c1749e465472c506305b2dd42d4cfa91725204893493']

,yyyymmdd,hour,city,appversion,isprefetched,userId,slotname,format,aduuid,epoch
0,20240901,21,Bangalore,8.30.0,true,65472c506305b2dd42d4cfa9,PostOrderOnTheWay1,OTWS0,cce203b4-35d2-4bd8-a816-8af48c1749e465472c5063...,1725205455685


In [47]:
CaptainSearch1= f"""

    select
        yyyymmdd,
        count(aduuid) request_count
    from
    (
    select 
        yyyymmdd,
        COALESCE(aduuid, adid) aduuid
    from 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    where  
        yyyymmdd >= '20240815'
        and yyyymmdd <= '20240902'
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        and slotname = 'CaptainSearch1'
        and eventName = 'Ad_Request'
    )
    group by 1
    order by 1   
    
"""

# Execute the query and get the result as a DataFrame
df_CaptainSearch1 = pd.read_sql(CaptainSearch1, connection)

In [48]:
df_CaptainSearch1

,yyyymmdd,request_count
0,20240815,466276
1,20240816,477285
2,20240817,481717
3,20240818,479732
4,20240819,881418
5,20240820,705964
6,20240821,692613
7,20240822,641955
8,20240823,653118
9,20240824,648026


In [50]:
df_CaptainSearch1.request_count.mean().round(0)

620157.0

### Check

In [57]:
check_query = f"""

WITH request as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        CASE WHEN appversion <= '8.28.0' THEN '8.28.0 >=' ELSE appversion END appversion,
        isprefetched,
        responsestatus,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '20240901' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '20240901' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        and slotname = 'PostOrderOnTheWay1'
        and eventName = 'Ad_Request'
        and appversion in ('8.30.0', '8.29.2')
    ),
    
    response as (

    SELECT     
        DISTINCT
        yyyymmdd,
        hh hour,
        city,
        CASE WHEN appversion <= '8.28.0' THEN '8.28.0 >=' ELSE appversion END appversion,
        isprefetched,
        responsestatus,
        userId,
        slotname,
        format,
        COALESCE(aduuid, adid) aduuid,
        epoch
    FROM 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
    
    WHERE  
        yyyymmdd >= '20240901' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '20240901' -- date_format({{end_date}},'%Y%m%d')
        and responseType = 'GAMBanner'
        and city = 'Bangalore'
        and slotname = 'PostOrderOnTheWay1'
        and eventName = 'Ad_Response'
        and appversion in ('8.30.0', '8.29.2')
    )
    
    SELECT
        request.yyyymmdd,
        request.slotname,
        request.appversion,
        request.isprefetched,
        COUNT(DISTINCT request.aduuid) request,
        COUNT(DISTINCT response.aduuid) response,
        COUNT(DISTINCT CASE WHEN response.responsestatus = 'true' THEN response.aduuid END) response_true
        
    FROM 
        request
    LEFT JOIN 
        response
        ON request.yyyymmdd = response.yyyymmdd
        AND request.city = response.city
        AND request.userId = response.userId
        AND request.slotname = response.slotname
        AND request.format = response.format
        AND request.aduuid = response.aduuid
        AND request.epoch <= response.epoch
        
    GROUP BY 1,2,3,4
    ORDER BY 1,2,3,4
    
"""

# Execute the query and get the result as a DataFrame
df_check_query = pd.read_sql(check_query, connection)

In [58]:
df_check_query

,yyyymmdd,slotname,appversion,isprefetched,request,response,response_true
0,20240901,PostOrderOnTheWay1,8.29.2,None,67813,10387,0
1,20240901,PostOrderOnTheWay1,8.30.0,true,230486,223501,48723
2,20240901,PostOrderOnTheWay1,8.30.0,None,168474,11891,0


In [73]:
start = '20240824' 
end = '20240903'

In [74]:
check_query = f"""

WITH exp_category AS (
    SELECT 
        yyyymmdd,
        customer_id,
        sample_category
    FROM 
        iceberg.experiments_internal.rapidex_user_customer_adSlots_v1_root 
    WHERE 
        yyyymmdd >= '{start}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end}' -- date_format({{end_date}},'%Y%m%d')
        AND experiment_name = 'CaptainSearchScreen_meme'
),

ads_tbl as(
select 
        event_yyyymmdd as yyyymmdd,
        userid,
        'meme_grp' AS test_group_type
from 
        canonical.iceberg_log_telemetry_ads_impressions_immutable
where
        yyyymmdd >= '{start}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end}' -- date_format({{end_date}},'%Y%m%d')
        and responsetype='GAMBanner'
        and city='Delhi'
        and eventname = 'Ad_Response'
        and regexp_like(metadata1,'unsee|Wahh|Expert|(mic drop)')
        and appversion = '8.30.0'
        
), 

fe_tbl AS (
    SELECT
        DISTINCT
        yyyymmdd,
        user_id AS user_id,
        city,
        -- service_name,
        fare_estimate_id
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
        yyyymmdd >= '{start}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end}' -- date_format({{end_date}},'%Y%m%d')
        AND city = 'Delhi'
        AND channel='android'
        --AND service_name = 'Auto'
),

rr_net AS (
    SELECT 
        yyyymmdd, 
        customer_id,
        estimate_id AS fare_estimate_id, 
        order_id, 
        order_status, 
        modified_order_status, 
        spd_fraud_flag,
        CASE 
            WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') 
            THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
        END AS time_spent_in_sec
    FROM 
        orders.order_logs_fact fact
    WHERE 
        yyyymmdd >= '{start}' -- date_format({{start_date}},'%Y%m%d')
        and yyyymmdd <= '{end}' -- date_format({{end_date}},'%Y%m%d')
        AND city_name = 'Delhi'
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium')
        AND channel_host='android'
),

merge AS (
    SELECT
       ads_tbl.test_group_type,
        exp_category.customer_id,
        exp_category.sample_category,
        fe_tbl.*,
        rr_net.order_id,
        rr_net.order_status,
        rr_net.modified_order_status,
        rr_net.spd_fraud_flag,
        rr_net.time_spent_in_sec
    FROM 
        exp_category

    LEFT JOIN 
        ads_tbl
        ON ads_tbl.yyyymmdd=exp_category.yyyymmdd
        AND ads_tbl.userid=exp_category.customer_id
        AND exp_category.sample_category='test'
        
    LEFT JOIN 
        fe_tbl
        ON exp_category.customer_id = fe_tbl.user_id 
        AND exp_category.yyyymmdd = fe_tbl.yyyymmdd
        
    LEFT JOIN rr_net
        ON fe_tbl.yyyymmdd = rr_net.yyyymmdd
        AND fe_tbl.user_id = rr_net.customer_id 
        AND fe_tbl.fare_estimate_id = rr_net.fare_estimate_id
)

SELECT
        yyyymmdd,
        sample_category,
        test_group_type,
        customers,
        fe_count,
        gross_count,
        net_count,
        COALESCE(TRY(gross_count*100.00/fe_count), 0) fe2rr,
        COALESCE(TRY(net_count*100.00/gross_count), 0) g2n,
        COALESCE(TRY(net_count*100.00/fe_count), 0) fe2net,
        
        median_cobrm_time,
        median_cobra_time,
        median_ocara_time,
        median_cancellation_time,
        
        COALESCE(TRY(cobrm*100.00/gross_count), 0) "% cobrm",
        COALESCE(TRY(cobra*100.00/gross_count), 0) "% cobra",
        COALESCE(TRY(ocara*100.00/gross_count), 0) "% ocara",
        cobrm, 
        cobra,
        ocara
    FROM    
        (
        SELECT
            
            yyyymmdd,
            sample_category,
            test_group_type,
            COUNT(DISTINCT user_id) customers,
            COUNT(DISTINCT fare_estimate_id) fe_count,
            COUNT(DISTINCT order_id) gross_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            
            approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN time_spent_in_sec END,0.50) as median_cobrm_time,
            approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN time_spent_in_sec END,0.50) as median_cobra_time,
            approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            approx_percentile(CASE WHEN modified_order_status in ('COBRA','COBRM','OCARA') THEN time_spent_in_sec END,0.50) as median_cancellation_time
            
        FROM 
            merge
        WHERE 
            yyyymmdd IS NOT NULL
        GROUP BY 1,2,3
        )
    ORDER BY 2,3 DESC,1
    
"""

# Execute the query and get the result as a DataFrame
df_check_query = pd.read_sql(check_query, connection)

In [75]:
df_check_query

,yyyymmdd,sample_category,test_group_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,% cobrm,% cobra,% ocara,cobrm,cobra,ocara
0,20240824,control,None,183870,606943,214489,137647,35.34,64.17,22.68,108.0,119.0,297.0,182,1.02,10.26,19.06,2180,22017,40889
1,20240825,control,None,158274,516718,173986,109621,33.67,63.01,21.21,109.0,119.0,291.0,180,1.41,10.12,19.58,2446,17616,34072
2,20240826,control,None,162718,514916,178417,118389,34.65,66.36,22.99,108.0,114.0,279.0,181,1.10,8.47,19.40,1966,15115,34616
3,20240827,control,None,201112,716051,258235,144327,36.06,55.89,20.16,105.0,129.0,298.0,169,1.15,17.59,17.72,2975,45418,45772
4,20240828,control,None,202389,733119,268862,144948,36.67,53.91,19.77,106.0,127.0,307.0,169,0.99,18.49,17.95,2675,49712,48259
5,20240829,control,None,212763,802070,281821,135340,35.14,48.02,16.87,106.0,133.0,309.0,170,1.28,20.71,17.36,3614,58352,48925
6,20240830,control,None,202460,698050,247287,154651,35.43,62.54,22.15,112.0,121.0,293.0,180,0.82,12.20,18.77,2021,30164,46425
7,20240831,control,None,192771,646099,221071,145852,34.22,65.98,22.57,109.0,114.0,297.0,187,0.92,8.92,19.46,2035,19711,43030
8,20240901,control,None,170503,571873,186903,118972,32.68,63.65,20.80,105.0,120.0,285.0,180,1.27,9.58,19.83,2368,17912,37064
9,20240902,control,None,209672,739614,260866,153964,35.27,59.02,20.82,105.0,124.0,294.0,174,0.90,14.31,18.75,2342,37331,48913


In [76]:
df_check_query[['yyyymmdd', 'sample_category', 'test_group_type', 'median_cobrm_time', 'median_cobra_time', 'median_ocara_time']]
# .to_clipboard(index=False)

,yyyymmdd,sample_category,test_group_type,median_cobrm_time,median_cobra_time,median_ocara_time
0,20240824,control,None,108.0,119.0,297.0
1,20240825,control,None,109.0,119.0,291.0
2,20240826,control,None,108.0,114.0,279.0
3,20240827,control,None,105.0,129.0,298.0
4,20240828,control,None,106.0,127.0,307.0
5,20240829,control,None,106.0,133.0,309.0
6,20240830,control,None,112.0,121.0,293.0
7,20240831,control,None,109.0,114.0,297.0
8,20240901,control,None,105.0,120.0,285.0
9,20240902,control,None,105.0,124.0,294.0


In [62]:
df_check_query

,yyyymmdd,sample_category,test_group_type,customers,fe_count,gross_count,net_count,fe2rr,g2n,fe2net,median_cobrm_time,median_cobra_time,median_ocara_time,median_cancellation_time,% cobrm,% cobra,% ocara,cobrm,cobra,ocara
0,20240829,control,None,212763,802070,281821,135340,35.14,48.02,16.87,107,134,310,171,1.28,20.71,17.36,3614,58352,48925
1,20240830,control,None,202460,698050,247287,154651,35.43,62.54,22.15,113,121,293,178,0.82,12.20,18.77,2021,30164,46425
2,20240831,control,None,192771,646099,221071,145852,34.22,65.98,22.57,109,114,297,187,0.92,8.92,19.46,2035,19711,43030
3,20240829,test,meme_grp,7156,49947,22561,5919,45.17,26.24,11.85,116,145,294,163,2.42,34.71,16.86,547,7830,3804
4,20240830,test,meme_grp,6050,36356,15929,6267,43.81,39.34,17.24,113,137,287,164,2.29,24.95,21.72,365,3974,3459
5,20240831,test,meme_grp,5685,33465,14631,5984,43.72,40.90,17.88,95,133,251,157,3.01,20.78,24.25,441,3041,3548
6,20240829,test,None,200857,732868,253018,125825,34.52,49.73,17.17,106,131,310,169,1.22,19.70,17.35,3096,49853,43888
7,20240830,test,None,192277,642995,224597,144050,34.93,64.14,22.40,104,119,297,180,0.72,11.31,18.60,1617,25404,41780
8,20240831,test,None,183100,599250,202565,137440,33.80,67.85,22.94,107,112,296,189,0.80,7.99,19.15,1626,16193,38796
